In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/UNI/M2_VMI/LabKD1'
os.makedirs(SAVE_DIR, exist_ok=True)
CKPT = f'{SAVE_DIR}/teacher_best.pt'

# QST 1 : Entrainement du teacher
sur CIFAR10, avec un split train/val/test, et compter les paramètres.

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision, torchvision.transforms as T
from torchvision.models import resnet50, ResNet50_Weights

device = "cuda" if torch.cuda.is_available() else "cpu"

# Données : CIFAR10 redimensionné à 224
mean, std = (0.485,0.456,0.406), (0.229,0.224,0.225)
tf_train = T.Compose([T.Resize(224), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(mean,std)])
tf_eval  = T.Compose([T.Resize(224), T.ToTensor(), T.Normalize(mean,std)])

full_train = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=tf_train)
full_val   = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=tf_eval)
test_set   = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=tf_eval)

# Split 45000 / 5000
g = torch.Generator().manual_seed(42)
idx_train, idx_val = random_split(range(50000), [45000, 5000], generator=g)
train_set = torch.utils.data.Subset(full_train, idx_train.indices)
val_set   = torch.utils.data.Subset(full_val,   idx_val.indices)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_set,   batch_size=256, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False, num_workers=2)

# Modèle teacher
teacher = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
teacher.fc = nn.Linear(teacher.fc.in_features, 10)
teacher = teacher.to(device)

# Nombre de paramètres
total = sum(p.numel() for p in teacher.parameters())
trainable = sum(p.numel() for p in teacher.parameters() if p.requires_grad)
print(f"Teacher ResNet50 : {total:,} paramètres ({trainable:,} entraînables)")

# Entraînement / évaluation
def evaluate(model, loader):
    model.eval(); correct = total_n = 0; loss_sum = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += nn.functional.cross_entropy(out, y, reduction="sum").item()
            correct += (out.argmax(1) == y).sum().item(); total_n += y.size(0)
    return loss_sum/total_n, correct/total_n

def train(model, epochs=5, lr=1e-3):
    opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    best, hist = 0, []
    for ep in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = nn.functional.cross_entropy(model(x), y)
            loss.backward(); opt.step()
        sched.step()
        vl, va = evaluate(model, val_loader)
        hist.append((vl, va))
        print(f"Epoch {ep+1}/{epochs} | val loss {vl:.4f} | val acc {va:.4f}")
        if va > best:
            best = va; torch.save(model.state_dict(), "teacher_best.pt")
    return hist

hist = train(teacher, epochs=5)

In [ ]:
torch.save(teacher.state_dict(), CKPT)
print(os.path.exists(CKPT), os.path.getsize(CKPT)/1e6, "Mo")

In [ ]:
tl, ta = evaluate(teacher, test_loader)
print(f"TEST : loss {tl:.4f} | acc {ta:.4f}")

# Strategie 1
## QST 2 : Entrainement de l'éleve
(pré-entraîné sur ImageNet) sur le jeu de données CIFAR10, tout en procédant à son évaluation.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

SAVE_DIR = '/content/drive/MyDrive/UNI/M2_VMI/LabKD1'
CKPT_S1 = f'{SAVE_DIR}/student_pre_best.pt'

# Même fonction que pour le teacher, avec le chemin de sauvegarde en paramètre
def train(model, ckpt, epochs=5, lr=1e-3):
    opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    best, hist = 0, []
    for ep in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = nn.functional.cross_entropy(model(x), y)
            loss.backward(); opt.step()
        sched.step()
        vl, va = evaluate(model, val_loader)
        hist.append((vl, va))
        print(f"Epoch {ep+1}/{epochs} | val loss {vl:.4f} | val acc {va:.4f}")
        if va > best:
            best = va
            torch.save(model.state_dict(), ckpt)
    return hist

# Student pré-entraîné ImageNet
student_pre = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
student_pre.fc = nn.Linear(student_pre.fc.in_features, 10)
student_pre = student_pre.to(device)

total = sum(p.numel() for p in student_pre.parameters())
trainable = sum(p.numel() for p in student_pre.parameters() if p.requires_grad)
print(f"Student ResNet18 : {total:,} paramètres ({trainable:,} entraînables)")

hist_s1 = train(student_pre, CKPT_S1, epochs=5)

# Test avec le meilleur checkpoint
student_pre.load_state_dict(torch.load(CKPT_S1, map_location=device))
sl, sa = evaluate(student_pre, test_loader)
print(f"TEST student : loss {sl:.4f} | acc {sa:.4f}")

# Comparaison avec le teacher
tp = sum(p.numel() for p in teacher.parameters())
print(f"Ratio paramètres teacher/student : {tp/total:.2f}x")

## QST 3 : Figer les paramètres du réseau teacher et entrainer l'élève
sous la supervision de l'enseignant, en utilisant la distillation de connaissances au niveau des scores.

In [ ]:
import copy

CKPT_S3 = f'{SAVE_DIR}/student_kd_scores_best.pt'

# Teacher gelé
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

def kd_scores_loss(s_logits, t_logits, y, alpha=0.5):
    ce  = nn.functional.cross_entropy(s_logits, y)
    mse = nn.functional.mse_loss(s_logits, t_logits)
    return (1 - alpha) * ce + alpha * mse, ce.item(), mse.item()

def train_kd(student, ckpt, epochs=5, lr=1e-3, alpha=0.5):
    opt = optim.SGD(student.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    best, hist = 0, []
    for ep in range(epochs):
        student.train(); teacher.eval()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                t_logits = teacher(x)
            s_logits = student(x)
            loss, ce, mse = kd_scores_loss(s_logits, t_logits, y, alpha)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        vl, va = evaluate(student, val_loader)
        hist.append((vl, va))
        print(f"Epoch {ep+1}/{epochs} | CE {ce:.4f} | MSE {mse:.4f} | val loss {vl:.4f} | val acc {va:.4f}")
        if va > best:
            best = va
            torch.save(student.state_dict(), ckpt)
    return hist

# Nouveau student pré-entraîné ImageNet (mêmes conditions de départ qu'à la question 2)
student_kd = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
student_kd.fc = nn.Linear(student_kd.fc.in_features, 10)
student_kd = student_kd.to(device)

hist_s3 = train_kd(student_kd, CKPT_S3, epochs=5, alpha=0.5)

student_kd.load_state_dict(torch.load(CKPT_S3, map_location=device))
kl, ka = evaluate(student_kd, test_loader)
print(f"TEST student KD scores : loss {kl:.4f} | acc {ka:.4f}")
print(f"Gain vs student sans KD : {100*(ka - sa):+.2f} points")

## QST 4 : Figer les paramètres du réseau teacher et entraîner le réseau élève
sous la supervision de l'enseignant, en utilisant la distillation de connaissances, tant au niveau des scores qu'au niveau des cartes de caractéristiques.

In [ ]:
CKPT_S4 = f'{SAVE_DIR}/student_kd_feat_best.pt'

def forward_feats(model, x):
    x = model.maxpool(model.relu(model.bn1(model.conv1(x))))
    x = model.layer1(x); x = model.layer2(x); x = model.layer3(x)
    f = model.layer4(x)                         # feature map finale
    logits = model.fc(torch.flatten(model.avgpool(f), 1))
    return f, logits

class Projector(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.proj = nn.Sequential(nn.Conv2d(c_in, c_out, 1, bias=False), nn.BatchNorm2d(c_out))
    def forward(self, x): return self.proj(x)

def train_kd_feat(student, projector, ckpt, epochs=5, lr=1e-3, alpha=0.5, beta=1.0):
    params = list(student.parameters()) + list(projector.parameters())
    opt = optim.SGD(params, lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    best, hist = 0, []
    for ep in range(epochs):
        student.train(); projector.train(); teacher.eval()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                t_feat, t_logits = forward_feats(teacher, x)
            s_feat, s_logits = forward_feats(student, x)
            ce   = nn.functional.cross_entropy(s_logits, y)
            mse_l = nn.functional.mse_loss(s_logits, t_logits)
            mse_f = nn.functional.mse_loss(projector(s_feat), t_feat)
            loss = (1 - alpha) * ce + alpha * mse_l + beta * mse_f
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        vl, va = evaluate(student, val_loader)
        hist.append((vl, va))
        print(f"Epoch {ep+1}/{epochs} | CE {ce.item():.4f} | MSE logits {mse_l.item():.4f} | MSE feat {mse_f.item():.4f} | val loss {vl:.4f} | val acc {va:.4f}")
        if va > best:
            best = va
            torch.save(student.state_dict(), ckpt)
    return hist

student_kd2 = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
student_kd2.fc = nn.Linear(student_kd2.fc.in_features, 10)
student_kd2 = student_kd2.to(device)
projector = Projector(512, 2048).to(device)

hist_s4 = train_kd_feat(student_kd2, projector, CKPT_S4, epochs=5, alpha=0.5, beta=1.0)

student_kd2.load_state_dict(torch.load(CKPT_S4, map_location=device))
fl, fa = evaluate(student_kd2, test_loader)
print(f"TEST student KD scores+features : loss {fl:.4f} | acc {fa:.4f}")
print(f"Gain vs sans KD : {100*(fa - sa):+.2f} pts | vs KD scores : {100*(fa - ka):+.2f} pts")

# Stratégie 2 : Distillation de connaissances utilisant un modèle élève non pré-entraîné
## QST 5 : Entraîner le réseau élève (non pré-entraîné sur ImageNet) à partir de zéro
sur le jeu de données CIFAR10, tout en procédant à son évaluation. Veiller à une répartition appropriée des données pour l'entraînement, la validation et le test.

In [ ]:
CKPT_S5 = f'{SAVE_DIR}/student_scratch_best.pt'
EPOCHS_SCRATCH = 15
LR_SCRATCH = 0.05

student_scratch = resnet18(weights=None)
student_scratch.fc = nn.Linear(student_scratch.fc.in_features, 10)
student_scratch = student_scratch.to(device)

total = sum(p.numel() for p in student_scratch.parameters())
print(f"Student ResNet18 from scratch : {total:,} paramètres")

hist_s5 = train(student_scratch, CKPT_S5, epochs=EPOCHS_SCRATCH, lr=LR_SCRATCH)

student_scratch.load_state_dict(torch.load(CKPT_S5, map_location=device))
cl, ca = evaluate(student_scratch, test_loader)
print(f"TEST student scratch : loss {cl:.4f} | acc {ca:.4f}")

## QST 6 : Figer les paramètres du réseau teacher et entraîner le réseau élève
sous la supervision de l'enseignant, en utilisant la distillation de connaissances au niveau des scores.

In [ ]:
CKPT_S6 = f'{SAVE_DIR}/student_scratch_kd_scores_best.pt'

student_scratch_kd = resnet18(weights=None)
student_scratch_kd.fc = nn.Linear(student_scratch_kd.fc.in_features, 10)
student_scratch_kd = student_scratch_kd.to(device)

hist_s6 = train_kd(student_scratch_kd, CKPT_S6,
                   epochs=EPOCHS_SCRATCH, lr=LR_SCRATCH, alpha=0.5)

student_scratch_kd.load_state_dict(torch.load(CKPT_S6, map_location=device))
kl6, ka6 = evaluate(student_scratch_kd, test_loader)
print(f"TEST scratch KD scores : loss {kl6:.4f} | acc {ka6:.4f}")
print(f"Gain vs scratch sans KD : {100*(ka6 - ca):+.2f} points")

sauvegarde

In [ ]:
import os
print(os.path.exists(CKPT_S6), os.path.getsize(CKPT_S6)/1e6, "Mo")   # ~45 Mo attendu

In [ ]:
import json
RES = f'{SAVE_DIR}/results.json'
results = json.load(open(RES)) if os.path.exists(RES) else {}

results.update({
    "teacher":              {"test_acc": ta},
    "student_pre":          {"test_acc": sa, "hist": hist_s1},
    "kd_scores":            {"test_acc": ka, "hist": hist_s3},
    "kd_scores_feat":       {"test_acc": fa, "hist": hist_s4},
    "scratch":              {"test_acc": ca, "hist": hist_s5},
    "scratch_kd_scores":    {"test_acc": ka6, "hist": hist_s6},
})
json.dump(results, open(RES, "w"), indent=2)

## QST 7 : Figer les paramètres du réseau teacher et entraîner le réseau élève sous la supervision du teacher
en utilisant la distillation de connaissances à la fois au niveau des scores et au niveau des cartes de caractéristiques (feature maps).

In [ ]:
import json

CKPT_S7 = f'{SAVE_DIR}/student_scratch_kd_feat_best.pt'
LAST_S7 = f'{SAVE_DIR}/student_scratch_kd_feat_last.pt'

def train_kd_feat_resume(student, projector, ckpt, last, epochs, lr, alpha=0.5, beta=1.0):
    params = list(student.parameters()) + list(projector.parameters())
    opt = optim.SGD(params, lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    best, hist, start = 0, [], 0

    if os.path.exists(last):
        st = torch.load(last, map_location=device)
        student.load_state_dict(st['student']); projector.load_state_dict(st['proj'])
        opt.load_state_dict(st['opt']); sched.load_state_dict(st['sched'])
        best, hist, start = st['best'], st['hist'], st['epoch'] + 1
        print(f"Reprise à l'epoch {start+1}")

    for ep in range(start, epochs):
        student.train(); projector.train(); teacher.eval()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                t_feat, t_logits = forward_feats(teacher, x)
            s_feat, s_logits = forward_feats(student, x)
            ce    = nn.functional.cross_entropy(s_logits, y)
            mse_l = nn.functional.mse_loss(s_logits, t_logits)
            mse_f = nn.functional.mse_loss(projector(s_feat), t_feat)
            loss = (1 - alpha) * ce + alpha * mse_l + beta * mse_f
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        vl, va = evaluate(student, val_loader)
        hist.append((vl, va))
        print(f"Epoch {ep+1}/{epochs} | CE {ce.item():.4f} | MSE logits {mse_l.item():.4f} | MSE feat {mse_f.item():.4f} | val loss {vl:.4f} | val acc {va:.4f}")
        if va > best:
            best = va
            torch.save(student.state_dict(), ckpt)
        torch.save({'epoch': ep, 'student': student.state_dict(), 'proj': projector.state_dict(),
                    'opt': opt.state_dict(), 'sched': sched.state_dict(),
                    'best': best, 'hist': hist}, last)
    return hist

student_scratch_kd2 = resnet18(weights=None)
student_scratch_kd2.fc = nn.Linear(student_scratch_kd2.fc.in_features, 10)
student_scratch_kd2 = student_scratch_kd2.to(device)
projector7 = Projector(512, 2048).to(device)

hist_s7 = train_kd_feat_resume(student_scratch_kd2, projector7, CKPT_S7, LAST_S7,
                               epochs=EPOCHS_SCRATCH, lr=LR_SCRATCH, alpha=0.5, beta=1.0)

student_scratch_kd2.load_state_dict(torch.load(CKPT_S7, map_location=device))
fl7, fa7 = evaluate(student_scratch_kd2, test_loader)
print(f"TEST scratch KD scores+features : loss {fl7:.4f} | acc {fa7:.4f}")
print(f"Gain vs scratch sans KD : {100*(fa7 - ca):+.2f} pts | vs scratch KD scores : {100*(fa7 - ka6):+.2f} pts")

RES = f'{SAVE_DIR}/results.json'
results = json.load(open(RES)) if os.path.exists(RES) else {}
results["scratch_kd_scores_feat"] = {"test_acc": fa7, "hist": hist_s7}
json.dump(results, open(RES, "w"), indent=2)

### à faire tourner en cas d'interruption de Google Colab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, torch, torch.nn as nn, torch.optim as optim
import torchvision, torchvision.transforms as T
from torch.utils.data import DataLoader, random_split
from torchvision.models import resnet18, resnet50, ResNet18_Weights

device = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_DIR = '/content/drive/MyDrive/UNI/M2_VMI/LabKD1'
CKPT_S7  = f'{SAVE_DIR}/student_scratch_kd_feat_best.pt'
LAST_S7  = f'{SAVE_DIR}/student_scratch_kd_feat_last.pt'
EPOCHS_SCRATCH, LR_SCRATCH = 15, 0.05

print(device, os.path.exists(LAST_S7), os.path.exists(f'{SAVE_DIR}/teacher_best.pt'))

In [ ]:
mean, std = (0.485,0.456,0.406), (0.229,0.224,0.225)
tf_train = T.Compose([T.Resize(224), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(mean,std)])
tf_eval  = T.Compose([T.Resize(224), T.ToTensor(), T.Normalize(mean,std)])

full_train = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=tf_train)
full_val   = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=tf_eval)
test_set   = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=tf_eval)

g = torch.Generator().manual_seed(42)
idx_train, idx_val = random_split(range(50000), [45000, 5000], generator=g)
train_set = torch.utils.data.Subset(full_train, idx_train.indices)
val_set   = torch.utils.data.Subset(full_val,   idx_val.indices)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_set,   batch_size=256, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False, num_workers=2)

In [ ]:
def evaluate(model, loader):
    model.eval(); correct = total_n = 0; loss_sum = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += nn.functional.cross_entropy(out, y, reduction="sum").item()
            correct += (out.argmax(1) == y).sum().item(); total_n += y.size(0)
    return loss_sum/total_n, correct/total_n

def forward_feats(model, x):
    x = model.maxpool(model.relu(model.bn1(model.conv1(x))))
    x = model.layer1(x); x = model.layer2(x); x = model.layer3(x)
    f = model.layer4(x)                         # feature map finale
    logits = model.fc(torch.flatten(model.avgpool(f), 1))
    return f, logits

class Projector(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.proj = nn.Sequential(nn.Conv2d(c_in, c_out, 1, bias=False), nn.BatchNorm2d(c_out))
    def forward(self, x): return self.proj(x)

def train_kd_feat(student, projector, ckpt, epochs=5, lr=1e-3, alpha=0.5, beta=1.0):
    params = list(student.parameters()) + list(projector.parameters())
    opt = optim.SGD(params, lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    best, hist = 0, []
    for ep in range(epochs):
        student.train(); projector.train(); teacher.eval()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                t_feat, t_logits = forward_feats(teacher, x)
            s_feat, s_logits = forward_feats(student, x)
            ce   = nn.functional.cross_entropy(s_logits, y)
            mse_l = nn.functional.mse_loss(s_logits, t_logits)
            mse_f = nn.functional.mse_loss(projector(s_feat), t_feat)
            loss = (1 - alpha) * ce + alpha * mse_l + beta * mse_f
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        vl, va = evaluate(student, val_loader)
        hist.append((vl, va))
        print(f"Epoch {ep+1}/{epochs} | CE {ce.item():.4f} | MSE logits {mse_l.item():.4f} | MSE feat {mse_f.item():.4f} | val loss {vl:.4f} | val acc {va:.4f}")
        if va > best:
            best = va
            torch.save(student.state_dict(), ckpt)
    return hist

def kd_scores_loss(s_logits, t_logits, y, alpha=0.5):
    ce  = nn.functional.cross_entropy(s_logits, y)
    mse = nn.functional.mse_loss(s_logits, t_logits)
    return (1 - alpha) * ce + alpha * mse, ce.item(), mse.item()

def train_kd(student, ckpt, epochs=5, lr=1e-3, alpha=0.5):
    opt = optim.SGD(student.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    best, hist = 0, []
    for ep in range(epochs):
        student.train(); teacher.eval()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                t_logits = teacher(x)
            s_logits = student(x)
            loss, ce, mse = kd_scores_loss(s_logits, t_logits, y, alpha)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        vl, va = evaluate(student, val_loader)
        hist.append((vl, va))
        print(f"Epoch {ep+1}/{epochs} | CE {ce:.4f} | MSE {mse:.4f} | val loss {vl:.4f} | val acc {va:.4f}")
        if va > best:
            best = va
            torch.save(student.state_dict(), ckpt)
    return hist

def train_kd_feat_resume(student, projector, ckpt, last, epochs, lr, alpha=0.5, beta=1.0):
    params = list(student.parameters()) + list(projector.parameters())
    opt = optim.SGD(params, lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    best, hist, start = 0, [], 0

    if os.path.exists(last):
        st = torch.load(last, map_location=device)
        student.load_state_dict(st['student']); projector.load_state_dict(st['proj'])
        opt.load_state_dict(st['opt']); sched.load_state_dict(st['sched'])
        best, hist, start = st['best'], st['hist'], st['epoch'] + 1
        print(f"Reprise à l'epoch {start+1}")

    for ep in range(start, epochs):
        student.train(); projector.train(); teacher.eval()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                t_feat, t_logits = forward_feats(teacher, x)
            s_feat, s_logits = forward_feats(student, x)
            ce    = nn.functional.cross_entropy(s_logits, y)
            mse_l = nn.functional.mse_loss(s_logits, t_logits)
            mse_f = nn.functional.mse_loss(projector(s_feat), t_feat)
            loss = (1 - alpha) * ce + alpha * mse_l + beta * mse_f
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        vl, va = evaluate(student, val_loader)
        hist.append((vl, va))
        print(f"Epoch {ep+1}/{epochs} | CE {ce.item():.4f} | MSE logits {mse_l.item():.4f} | MSE feat {mse_f.item():.4f} | val loss {vl:.4f} | val acc {va:.4f}")
        if va > best:
            best = va
            torch.save(student.state_dict(), ckpt)
        torch.save({'epoch': ep, 'student': student.state_dict(), 'proj': projector.state_dict(),
                    'opt': opt.state_dict(), 'sched': sched.state_dict(),
                    'best': best, 'hist': hist}, last)
    return hist


In [ ]:
teacher = resnet50()
teacher.fc = nn.Linear(teacher.fc.in_features, 10)
teacher.load_state_dict(torch.load(f'{SAVE_DIR}/teacher_best.pt', map_location=device))
teacher = teacher.to(device).eval()
for p in teacher.parameters():
    p.requires_grad = False

### Reprise

In [ ]:
student_scratch_kd2 = resnet18(weights=None)
student_scratch_kd2.fc = nn.Linear(512, 10)
student_scratch_kd2 = student_scratch_kd2.to(device)
projector7 = Projector(512, 2048).to(device)

hist_s7 = train_kd_feat_resume(student_scratch_kd2, projector7, CKPT_S7, LAST_S7,
                               epochs=EPOCHS_SCRATCH, lr=LR_SCRATCH, alpha=0.5, beta=1.0)

In [ ]:
student_scratch_kd2.load_state_dict(torch.load(CKPT_S7, map_location=device))
fl7, fa7 = evaluate(student_scratch_kd2, test_loader)
print(f"TEST scratch KD scores+features : loss {fl7:.4f} | acc {fa7:.4f}")

RES = f'{SAVE_DIR}/results.json'
results = json.load(open(RES)) if os.path.exists(RES) else {}
results["scratch_kd_scores_feat"] = {"test_acc": fa7, "hist": hist_s7}
json.dump(results, open(RES, "w"), indent=2)

# Comparaisons
## QST 8 : Comparer Stratégie 1 et 2

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt

RES = f'{SAVE_DIR}/results.json'
results = json.load(open(RES)) if os.path.exists(RES) else {}


fallback = {"teacher": 0.9553, "student_pre": 0.9347, "kd_scores": 0.9438,
            "kd_scores_feat": 0.9418, "scratch": 0.8871,
            "scratch_kd_scores": None,          # à remplir avec ton résultat Q6
            "scratch_kd_scores_feat": 0.9228}
acc = {k: results.get(k, {}).get("test_acc", fallback[k]) for k in fallback}
assert acc["scratch_kd_scores"] is not None, "Renseigne la test acc de Q6 dans fallback"

rows = []
for strat, base, keys in [
    ("Stratégie 1 (pré-entraîné)", "student_pre", ["student_pre", "kd_scores", "kd_scores_feat"]),
    ("Stratégie 2 (from scratch)", "scratch",     ["scratch", "scratch_kd_scores", "scratch_kd_scores_feat"]),
]:
    for k, label in zip(keys, ["Sans KD", "KD scores", "KD scores + features"]):
        rows.append({"Stratégie": strat, "Méthode": label,
                     "Test acc (%)": 100*acc[k],
                     "Gain vs sans KD (pts)": 100*(acc[k] - acc[base]),
                     "Écart au teacher (pts)": 100*(acc[k] - acc["teacher"])})
df = pd.DataFrame(rows).round(2)
print(df.to_string(index=False))
df.to_csv(f'{SAVE_DIR}/comparaison_strategies.csv', index=False)


hist_keys = {"S1 sans KD": "student_pre", "S1 KD scores": "kd_scores", "S1 KD scores+feat": "kd_scores_feat",
             "S2 sans KD": "scratch", "S2 KD scores": "scratch_kd_scores", "S2 KD scores+feat": "scratch_kd_scores_feat"}
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for label, k in hist_keys.items():
    h = results.get(k, {}).get("hist")
    if h is None: continue
    a = ax[0] if label.startswith("S1") else ax[1]
    a.plot(range(1, len(h)+1), [x[1] for x in h], marker="o", label=label)
ax[0].set_title("Stratégie 1 : val acc"); ax[1].set_title("Stratégie 2 : val acc")
for a in ax: a.set_xlabel("Epoch"); a.set_ylabel("Val acc"); a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.savefig(f'{SAVE_DIR}/courbes_strategies.png', dpi=150); plt.show()

## QST 9 : Comparer différentes métriques de distance
entre l’enseignant et l’élève (autres que la MSE)

In [ ]:
import torch.nn.functional as F

def dist_logits(name, s, t, T=4.0):
    if name == "mse":    return F.mse_loss(s, t)
    if name == "l1":     return F.l1_loss(s, t)
    if name == "huber":  return F.smooth_l1_loss(s, t)
    if name == "cosine": return (1 - F.cosine_similarity(s, t, dim=1)).mean()
    if name == "kl":
        return F.kl_div(F.log_softmax(s/T, 1), F.softmax(t/T, 1), reduction="batchmean") * T * T
    raise ValueError(name)

def dist_feat(name, s, t):
    if name == "mse":    return F.mse_loss(s, t)
    if name == "l1":     return F.l1_loss(s, t)
    if name == "huber":  return F.smooth_l1_loss(s, t)
    if name in ("cosine", "kl"):
        return (1 - F.cosine_similarity(s, t, dim=1)).mean()
    raise ValueError(name)

def run_kd(metric, use_feat, epochs=5, lr=1e-3, alpha=0.5, beta=1.0):
    tag = f"q9_{metric}_{'feat' if use_feat else 'scores'}"
    ckpt = f'{SAVE_DIR}/{tag}_best.pt'

    student = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    student.fc = nn.Linear(512, 10); student = student.to(device)
    proj = Projector(512, 2048).to(device) if use_feat else None
    params = list(student.parameters()) + (list(proj.parameters()) if use_feat else [])
    opt = optim.SGD(params, lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, epochs)

    best, hist = 0, []
    for ep in range(epochs):
        student.train(); teacher.eval()
        if proj: proj.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                t_feat, t_logits = forward_feats(teacher, x)
            s_feat, s_logits = forward_feats(student, x)
            loss = (1 - alpha) * F.cross_entropy(s_logits, y) + alpha * dist_logits(metric, s_logits, t_logits)
            if use_feat:
                loss = loss + beta * dist_feat(metric, proj(s_feat), t_feat)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        vl, va = evaluate(student, val_loader); hist.append((vl, va))
        print(f"[{tag}] epoch {ep+1}/{epochs} | val loss {vl:.4f} | val acc {va:.4f}")
        if va > best:
            best = va; torch.save(student.state_dict(), ckpt)

    student.load_state_dict(torch.load(ckpt, map_location=device))
    tl, ta_ = evaluate(student, test_loader)
    print(f"[{tag}] TEST acc {ta_:.4f}")
    return tag, {"test_acc": ta_, "hist": hist}

RES = f'{SAVE_DIR}/results.json'
results = json.load(open(RES)) if os.path.exists(RES) else {}

# Série 1 : scores seuls. Série 2 : scores + features
configs = [(m, False) for m in ["l1", "huber", "cosine", "kl"]]


for metric, use_feat in configs:
    tag = f"q9_{metric}_{'feat' if use_feat else 'scores'}"
    if tag in results:
        print("déjà fait :", tag); continue
    tag, res = run_kd(metric, use_feat)
    results[tag] = res
    json.dump(results, open(RES, "w"), indent=2)

### à executer en cas d'interuption de Google Colab

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, torch, torch.nn as nn, torch.optim as optim, torch.nn.functional as F
import torchvision, torchvision.transforms as T
import pandas as pd
from torch.utils.data import DataLoader, random_split
from torchvision.models import resnet18, resnet50, ResNet18_Weights

device = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_DIR = '/content/drive/MyDrive/UNI/M2_VMI/LabKD1'
RES = f'{SAVE_DIR}/results.json'
print("GPU :", device, "| teacher :", os.path.exists(f'{SAVE_DIR}/teacher_best.pt'))


mean, std = (0.485,0.456,0.406), (0.229,0.224,0.225)
tf_train = T.Compose([T.Resize(224), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(mean,std)])
tf_eval  = T.Compose([T.Resize(224), T.ToTensor(), T.Normalize(mean,std)])
full_train = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=tf_train)
full_val   = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=tf_eval)
test_set   = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=tf_eval)
g = torch.Generator().manual_seed(42)
idx_train, idx_val = random_split(range(50000), [45000, 5000], generator=g)
train_set = torch.utils.data.Subset(full_train, idx_train.indices)
val_set   = torch.utils.data.Subset(full_val,   idx_val.indices)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_set,   batch_size=256, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False, num_workers=2)


def evaluate(model, loader):
    model.eval(); correct = n = 0; loss_sum = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += F.cross_entropy(out, y, reduction="sum").item()
            correct += (out.argmax(1) == y).sum().item(); n += y.size(0)
    return loss_sum/n, correct/n

def forward_feats(model, x):
    x = model.maxpool(model.relu(model.bn1(model.conv1(x))))
    x = model.layer1(x); x = model.layer2(x); x = model.layer3(x)
    f = model.layer4(x)
    return f, model.fc(torch.flatten(model.avgpool(f), 1))

class Projector(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.proj = nn.Sequential(nn.Conv2d(c_in, c_out, 1, bias=False), nn.BatchNorm2d(c_out))
    def forward(self, x): return self.proj(x)

def dist_logits(name, s, t, T=4.0):
    if name == "mse":    return F.mse_loss(s, t)
    if name == "l1":     return F.l1_loss(s, t)
    if name == "huber":  return F.smooth_l1_loss(s, t)
    if name == "cosine": return (1 - F.cosine_similarity(s, t, dim=1)).mean()
    if name == "kl":
        return F.kl_div(F.log_softmax(s/T, 1), F.softmax(t/T, 1), reduction="batchmean") * T * T
    raise ValueError(name)

def dist_feat(name, s, t):
    if name == "mse":    return F.mse_loss(s, t)
    if name == "l1":     return F.l1_loss(s, t)
    if name == "huber":  return F.smooth_l1_loss(s, t)
    return (1 - F.cosine_similarity(s, t, dim=1)).mean()

def run_kd(metric, use_feat, epochs=5, lr=1e-3, alpha=0.5, beta=1.0):
    tag = f"q9_{metric}_{'feat' if use_feat else 'scores'}"
    ckpt = f'{SAVE_DIR}/{tag}_best.pt'
    student = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    student.fc = nn.Linear(512, 10); student = student.to(device)
    proj = Projector(512, 2048).to(device) if use_feat else None
    params = list(student.parameters()) + (list(proj.parameters()) if use_feat else [])
    opt = optim.SGD(params, lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    best, hist = 0, []
    for ep in range(epochs):
        student.train(); teacher.eval()
        if proj: proj.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                t_feat, t_logits = forward_feats(teacher, x)
            s_feat, s_logits = forward_feats(student, x)
            loss = (1 - alpha) * F.cross_entropy(s_logits, y) + alpha * dist_logits(metric, s_logits, t_logits)
            if use_feat:
                loss = loss + beta * dist_feat(metric, proj(s_feat), t_feat)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        vl, va = evaluate(student, val_loader); hist.append((vl, va))
        print(f"[{tag}] epoch {ep+1}/{epochs} | val loss {vl:.4f} | val acc {va:.4f}")
        if va > best:
            best = va; torch.save(student.state_dict(), ckpt)
    student.load_state_dict(torch.load(ckpt, map_location=device))
    _, ta_ = evaluate(student, test_loader)
    print(f"[{tag}] TEST acc {ta_:.4f}")
    return tag, {"test_acc": ta_, "hist": hist}


teacher = resnet50()
teacher.fc = nn.Linear(teacher.fc.in_features, 10)
teacher.load_state_dict(torch.load(f'{SAVE_DIR}/teacher_best.pt', map_location=device))
teacher = teacher.to(device).eval()
for p in teacher.parameters(): p.requires_grad = False
print("Teacher test acc :", evaluate(teacher, test_loader)[1])


results = json.load(open(RES)) if os.path.exists(RES) else {}
configs = [(m, False) for m in ["l1", "huber", "cosine", "kl"]]
for metric, use_feat in configs:
    tag = f"q9_{metric}_{'feat' if use_feat else 'scores'}"
    if tag in results:
        print("déjà fait :", tag); continue
    tag, res = run_kd(metric, use_feat)
    results[tag] = res
    json.dump(results, open(RES, "w"), indent=2)

# --- Tableau final
ref = {"MSE (référence)": 0.9438}
rows = [{"Distance": k, "Test acc (%)": 100*v} for k, v in ref.items()]
for k, v in results.items():
    if k.startswith("q9_") and k.endswith("_scores"):
        rows.append({"Distance": k.split("_")[1], "Test acc (%)": 100*v["test_acc"]})
df9 = pd.DataFrame(rows)
df9["Gain vs sans KD (pts)"] = df9["Test acc (%)"] - 93.47
print(df9.round(2).to_string(index=False))
df9.to_csv(f'{SAVE_DIR}/q9_distances.csv', index=False)

Mounted at /content/drive
GPU : cuda | teacher : True


100%|██████████| 170M/170M [20:03<00:00, 142kB/s]


Teacher test acc : 0.9553
déjà fait : q9_l1_scores
déjà fait : q9_huber_scores
déjà fait : q9_cosine_scores
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 194MB/s]


[q9_kl_scores] epoch 1/5 | val loss 0.2233 | val acc 0.9200
[q9_kl_scores] epoch 2/5 | val loss 0.1817 | val acc 0.9360
[q9_kl_scores] epoch 3/5 | val loss 0.1589 | val acc 0.9430
[q9_kl_scores] epoch 4/5 | val loss 0.1533 | val acc 0.9456
[q9_kl_scores] epoch 5/5 | val loss 0.1522 | val acc 0.9462
[q9_kl_scores] TEST acc 0.9435
       Distance  Test acc (%)  Gain vs sans KD (pts)
MSE (référence)         94.38                   0.91
             l1         93.02                  -0.45
          huber         92.75                  -0.72
         cosine         92.68                  -0.79
             kl         94.35                   0.88


## QST 10 : Répéter les étapes de la première partie en utilisant votre réseau teacher personnalisé.
Testez différents réseaux élève, qui doivent être des versions réduites du réseau teacher (par exemple, /2, /3, /4,
etc.)

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, torch, torch.nn as nn, torch.optim as optim, torch.nn.functional as F
import torchvision, torchvision.transforms as T
import pandas as pd
from torch.utils.data import DataLoader, random_split

device = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_DIR = '/content/drive/MyDrive/UNI/M2_VMI/LabKD1'
RES = f'{SAVE_DIR}/results.json'
EPOCHS, LR, ALPHA, BETA = 20, 0.05, 0.5, 1.0

mean, std = (0.4914,0.4822,0.4465), (0.2470,0.2435,0.2616)
tf_train = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(mean,std)])
tf_eval  = T.Compose([T.ToTensor(), T.Normalize(mean,std)])
full_train = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=tf_train)
full_val   = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=tf_eval)
test_set   = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=tf_eval)
g = torch.Generator().manual_seed(42)
idx_train, idx_val = random_split(range(50000), [45000, 5000], generator=g)
train_loader = DataLoader(torch.utils.data.Subset(full_train, idx_train.indices), batch_size=128, shuffle=True, num_workers=2)
val_loader   = DataLoader(torch.utils.data.Subset(full_val,   idx_val.indices),   batch_size=256, num_workers=2)
test_loader  = DataLoader(test_set, batch_size=256, num_workers=2)

class CustomCNN(nn.Module):
    def __init__(self, w=64, nc=10):
        super().__init__()
        ch, layers, c_in = [w, 2*w, 4*w, 8*w], [], 3
        for c in ch:
            layers += [nn.Conv2d(c_in, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(True),
                       nn.Conv2d(c, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(True),
                       nn.MaxPool2d(2)]
            c_in = c
        self.features = nn.Sequential(*layers)
        self.fc = nn.Linear(ch[-1], nc)
        self.out_ch = ch[-1]
    def forward_feats(self, x):
        f = self.features(x)
        return f, self.fc(f.mean((2, 3)))
    def forward(self, x): return self.forward_feats(x)[1]

class Projector(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.proj = nn.Sequential(nn.Conv2d(c_in, c_out, 1, bias=False), nn.BatchNorm2d(c_out))
    def forward(self, x): return self.proj(x)

def evaluate(model, loader):
    model.eval(); correct = n = 0; loss_sum = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += F.cross_entropy(out, y, reduction="sum").item()
            correct += (out.argmax(1) == y).sum().item(); n += y.size(0)
    return loss_sum/n, correct/n

def n_params(m): return sum(p.numel() for p in m.parameters())

def fit(model, tag, mode="none", teacher_c=None):
    ckpt = f'{SAVE_DIR}/{tag}_best.pt'
    proj = Projector(model.out_ch, teacher_c.out_ch).to(device) if mode == "feat" else None
    params = list(model.parameters()) + (list(proj.parameters()) if proj else [])
    opt = optim.SGD(params, lr=LR, momentum=0.9, weight_decay=5e-4, nesterov=True)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
    best, hist = 0, []
    for ep in range(EPOCHS):
        model.train()
        if proj: proj.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            s_feat, s_logits = model.forward_feats(x)
            loss = F.cross_entropy(s_logits, y)
            if mode != "none":
                with torch.no_grad():
                    t_feat, t_logits = teacher_c.forward_feats(x)
                loss = (1 - ALPHA) * loss + ALPHA * F.mse_loss(s_logits, t_logits)
                if mode == "feat":
                    loss = loss + BETA * F.mse_loss(proj(s_feat), t_feat)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        vl, va = evaluate(model, val_loader); hist.append((vl, va))
        print(f"[{tag}] epoch {ep+1}/{EPOCHS} | val loss {vl:.4f} | val acc {va:.4f}")
        if va > best:
            best = va; torch.save(model.state_dict(), ckpt)
    model.load_state_dict(torch.load(ckpt, map_location=device))
    _, ta = evaluate(model, test_loader)
    print(f"[{tag}] TEST acc {ta:.4f}")
    return {"test_acc": ta, "hist": hist, "params": n_params(model)}

results = json.load(open(RES)) if os.path.exists(RES) else {}
def save(tag, res):
    results[tag] = res
    json.dump(results, open(RES, "w"), indent=2)

# Teacher
teacher_c = CustomCNN(64).to(device)
print(f"Teacher custom : {n_params(teacher_c):,} paramètres")
save("q10_teacher", fit(teacher_c, "q10_teacher"))
teacher_c.eval()
for p in teacher_c.parameters(): p.requires_grad = False

# Eleves /2, /3, /4
for div, w in [(2, 32), (3, 21), (4, 16)]:
    for mode in ["none", "scores", "feat"]:
        tag = f"q10_s{div}_{mode}"
        student = CustomCNN(w).to(device)
        save(tag, fit(student, tag, mode, teacher_c))

# Tableau
rows = [{"Modèle": "Teacher (w=64)", "Méthode": "-", "Paramètres": results["q10_teacher"]["params"],
         "Test acc (%)": 100*results["q10_teacher"]["test_acc"], "Gain vs sans KD (pts)": None}]
names = {"none": "Sans KD", "scores": "KD scores", "feat": "KD scores+features"}
for div in [2, 3, 4]:
    base = results[f"q10_s{div}_none"]["test_acc"]
    for mode in ["none", "scores", "feat"]:
        r = results[f"q10_s{div}_{mode}"]
        rows.append({"Modèle": f"Student /{div}", "Méthode": names[mode], "Paramètres": r["params"],
                     "Test acc (%)": 100*r["test_acc"], "Gain vs sans KD (pts)": 100*(r["test_acc"] - base)})
df10 = pd.DataFrame(rows).round(2)
print(df10.to_string(index=False))
df10.to_csv(f'{SAVE_DIR}/q10_resultats.csv', index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Teacher custom : 4,692,426 paramètres
[q10_teacher] epoch 1/20 | val loss 1.4447 | val acc 0.4978
[q10_teacher] epoch 2/20 | val loss 0.8869 | val acc 0.6876
[q10_teacher] epoch 3/20 | val loss 0.7956 | val acc 0.7250
[q10_teacher] epoch 4/20 | val loss 0.6435 | val acc 0.7624
[q10_teacher] epoch 5/20 | val loss 0.7422 | val acc 0.7526
[q10_teacher] epoch 6/20 | val loss 0.7067 | val acc 0.7654
[q10_teacher] epoch 7/20 | val loss 0.6279 | val acc 0.7806
[q10_teacher] epoch 8/20 | val loss 0.4519 | val acc 0.8442
[q10_teacher] epoch 9/20 | val loss 0.4835 | val acc 0.8318
[q10_teacher] epoch 10/20 | val loss 0.4294 | val acc 0.8536
[q10_teacher] epoch 11/20 | val loss 0.3837 | val acc 0.8726
[q10_teacher] epoch 12/20 | val loss 0.4022 | val acc 0.8680
[q10_teacher] epoch 13/20 | val loss 0.4044 | val acc 0.8688
[q10_teacher] epoch 14/20 | val loss 0.3442 | val